---
title: Alignment Metrics
subtitle: A view of alignment performance
date: "9999-12-31"
edit_url: null
---

In [4]:
import altair as alt
from harpy.report.html import StatsBox, print_html
from harpy.report.plots import SafeRender
from harpy.report.tables import ITable
from harpy.report.utilities import StopExecution
import os
import polars as pl
from pathlib import Path
_ = pl.Config.set_tbl_hide_column_data_types()


In [ ]:
indir = "reports/data"
platform = "haplotagging"


In [ ]:
keys_to_keep = set([
    "percentage_of_properly_paired_reads_(%)",
    "pairs_on_different_chromosomes",
    "reads_mapped_%",
    "reads_QC_failed_%",
    "error_rate",
    "supplementary_alignments",
    "raw_total_sequences",
    "insert_size_average",
    'self_&_mate_mapped_%'
])

In [ ]:
def get_val(x: str):
    '''Return the value of the line `x` as an int'''
    return int(x.strip().split(": ")[-1])

def parse_markdup(filename):
    '''Parses a samtools markdup output log file and returns the percent of duplicates as a tuple (pcr, optical)'''
    pcr = 0
    optical = 0
    reads = 0
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith("READ:"):
                reads = get_val(line)
            if line.startswith("DUPLICATE PAIR:") or line.startswith("DUPLICATE SINGLE:"):
                pcr += get_val(line)
            if line.startswith("DUPLICATE PAIR OPTICAL:") or line.startswith("DUPLICATE SINGLE OPTICAL:"):
                optical += get_val(line)
    if reads <= 0:
        return 0.0, 0.0
    pcr_perc = round(((pcr-optical) / reads) * 100, 2)
    optical_perc = round((optical / reads) * 100, 2)
    return pcr_perc, optical_perc

def parse_SN(filecontents: list[str]):
    """Parse `SN` rows from samtools stats output into a normalized dict.
    Also extract samtools and htslib versions if present on the header line.
    Returns a tuple: (parsed_data, samtools_version, htslib_version)
    """
    parsed_data = {}
    for line in filecontents:
        if not line.startswith("SN"):
            if line.startswith("# First Fragment Qualities."):
                break
            continue
        sections = line.split("\t")
        if len(sections) < 3:
            continue
        field = sections[1].strip()[:-1].replace(" ", "_").lstrip()
        try:
            value = float(sections[2].strip())
        except ValueError:
            continue
        parsed_data[field] = value
    return parsed_data


In [ ]:
def parse_samtools_stats(indir: str, datatype: str):
    """Find Samtools stats logs and parse their data"""
    statsdir = os.path.join(indir, "samtools_stats")
    markdupdir = os.path.join(indir, "markdup")
    samtools_stats = {}
    
    for f in Path(statsdir).glob(f"*{datatype}.stats"):
        samplename = f.stem.removesuffix(f".{datatype}")
        file_contents = f.read_text().splitlines()
        parsed_data = parse_SN(file_contents)
        pcr, optical = parse_markdup(os.path.join(markdupdir, f"{samplename}.markdup"))

        if len(parsed_data) > 0:
            # Work out some percentages
            if "raw_total_sequences" in parsed_data:
                seqs = parsed_data.get("sequences", 0)
                mapped_and_paired = parsed_data.get("reads_mapped_and_paired", 0)
                parsed_data['self_&_mate_mapped_%'] = round((mapped_and_paired / seqs) * 100, 1) if seqs > 0 else 0
                parsed_data['error_rate'] = round(parsed_data['error_rate'] * 100, 5)
                for k in list(parsed_data.keys()):
                    if k.startswith("reads_") and k != "raw_total_sequences" and parsed_data["raw_total_sequences"] > 0:
                        parsed_data[f"{k}_%"] = round((parsed_data[k] / parsed_data["raw_total_sequences"]) * 100, 2)
                        del parsed_data[k]
            samtools_stats[samplename] = {
                key.replace("_", " ").replace(" and ", " & ").replace("reads", "").title().strip():
                parsed_data.get(key, 0)
                for key in keys_to_keep
            }
            samtools_stats[samplename]["% Duplicates (PCR)"] = pcr
            samtools_stats[samplename]["% Duplicates (Opt)"] = optical
    if len(samtools_stats) == 0:
        return pl.DataFrame()
    
    df = pl.DataFrame([{"Sample": sample, **cols} for sample, cols in samtools_stats.items()]).rename({
        "Self & Mate Mapped %": "% Self+Mate Mapped",
        "Qc Failed %": "% QC Failed",
        "Raw Total Sequences": "Total Reads",
        "Insert Size Average": "Average Insert (bp)",
        "Percentage Of Properly Paired  (%)": "% Properly Paired",
        "Pairs On Different Chromosomes": "Diff Chromosome",
        "Mapped %": "% Mapped",
        "Error Rate": "Error Rate (%)",
        "Supplementary Alignments": "Supp Alignments"    
    }).select([
        "Sample",
        "Total Reads",
        "% Mapped",
        "% Properly Paired",
        "% Self+Mate Mapped",
        "% Duplicates (PCR)",
        "% Duplicates (Opt)",
        "Average Insert (bp)",
        "% QC Failed",
        "Diff Chromosome",
        "Error Rate (%)",
        "Supp Alignments",
    ])
    df = df.with_columns(
        pl.col("Average Insert (bp)").round(0).cast(pl.Int32),
        pl.col("Total Reads").cast(pl.Int64),
        pl.col("Diff Chromosome").cast(pl.Int64),
        pl.col("Supp Alignments").cast(pl.Int64)
    )
    return df


In [ ]:
dat = parse_samtools_stats(indir, "raw")
if len(dat) == 0:
    print_html("No statsfiles were found, or were unable to be properly parsed. Skipping report.")
    raise StopExecution


## Unprocessed (raw) Alignments
The data presented below show the metrics for the alignments prior to duplicate identification, unmapped removal, quality filtering, etc. The summary boxes are averages across all the samples, and their standard deviations.

In [ ]:
_pp = dat["% Properly Paired"].replace(0, None).mean() or 0
_ppstd = dat["% Properly Paired"].replace(0, None).std() or 0

_mapped = dat["% Mapped"].replace(0, None).mean() or 0
_mappedstd = dat["% Mapped"].replace(0, None).std() or 0

#_smm = dat["% Self+Mate Mapped"].replace(0, None).mean() or 0
#_smmstd = dat["% Self+Mate Mapped"].replace(0, None).std() or 0

_isize = round(dat['Average Insert (bp)'].replace(0, None).mean() or 0)
_isizestd = round(dat['Average Insert (bp)'].replace(0, None).std() or 0)

_err = dat["Error Rate (%)"].replace(0, None).mean() or 0
_errstd = dat["Error Rate (%)"].replace(0, None).std() or 0

_pcr = dat["% Duplicates (PCR)"].mean() or 0
_pcrstd = dat["% Duplicates (PCR)"].std() or 0

_opt = dat["% Duplicates (Opt)"].mean() or 0
_optstd = dat["% Duplicates (Opt)"].std() or 0

(
    StatsBox()
    .add(len(dat), "Samples")
    .conditional(_pp, "Properly Paired", 80, plus_minus=_ppstd, add_percent=True, digits = 1)
    .conditional(_mapped, "Mapped", 70, plus_minus=_mappedstd, add_percent=True, digits = 1)
    .conditional(_err, "Error Rate", 10, plus_minus=_errstd, lower_bad = False, add_percent=True, digits = 3)
    .conditional(_pcr, "PCR Duplicate Rate", 40, plus_minus=_pcrstd, lower_bad = False, add_percent=True, digits = 2)
    .conditional(_opt, "Optical Duplicate Rate", 40, plus_minus=_optstd, lower_bad = False, add_percent=True, digits = 2)
    .add(_isize, "Insert Size", plus_minus=_isizestd, units = "bp")
).render()

:::{important} Duplicates
:class: dropdown
:open: true
Duplicates were marked after removing unmapped reads (unless `harpy align` ran with `-u`), which would mean the duplicate percents reported below would not include unmapped reads. The flow cell distance to determine optical duplicates was inferred from the sequencer ID at the start of sequence IDs.
:::

In [ ]:
ITable(dat, filename = "align.raw.stats", fixedcols = 1).render()

## Processed Alignments
The data presented below show the metrics for the alignments following post-processing, which includes duplicate identification, unmapped removal (unless configured otherwise), quality filtering, etc. The summary boxes are averages across all the samples, and their standard deviations.

In [ ]:
dat_filt = parse_samtools_stats(indir, "filtered")
if len(dat_filt) == 0:
    print_html("No statsfiles were found, or were unable to be properly parsed. Skipping report.")
    raise StopExecution
dat_filt = dat_filt.select(pl.col("*").exclude("% Duplicates (PCR)", "% Duplicates (Opt)"))


In [ ]:
_pp = dat_filt["% Properly Paired"].replace(0, None).mean() or 0
_ppstd = dat_filt["% Properly Paired"].replace(0, None).std() or 0

_mapped = dat_filt["% Mapped"].replace(0, None).mean() or 0
_mappedstd = dat_filt["% Mapped"].replace(0, None).std() or 0

_isize = round(dat_filt['Average Insert (bp)'].replace(0, None).mean() or 0)
_isizestd = round(dat_filt['Average Insert (bp)'].replace(0, None).std() or 0)

_err = dat_filt["Error Rate (%)"].replace(0, None).mean() or 0
_errstd = dat_filt["Error Rate (%)"].replace(0, None).std() or 0

(
    StatsBox()
    .add(len(dat_filt), "Samples")
    .conditional(_pp, "Properly Paired", 80, plus_minus=_ppstd, add_percent=True, digits = 1)
    .conditional(_mapped, "Mapped", 70, plus_minus=_mappedstd, add_percent=True, digits = 1)
    .conditional(_err, "Error Rate", 10, plus_minus=_errstd, lower_bad = False, add_percent=True, digits = 3)
    .add(_isize, "Insert Size", plus_minus=_isizestd, units = "bp")
).render()

:::{hint} Usable Data
:class: dropdown
:open: true
All the values below are provided **ignoring reads marked as duplicates**, meaning these should be views as the properties of "usable data".
:::

In [ ]:
ITable(dat_filt, filename = "align.processed.stats", fixedcols = 1).render()

## Duplication Rate
Duplication rate can vary significantly between libraries and preparation methods and it can often been a large pain-point in sequencing success. Below is a plot of duplication rate across samples, separated by PCR[^1] and Optical[^2] duplicates. Keep in mind these are kernel density estimates, so the Y-axis reflects density and not sample proportion. 

[^1]: PCR duplicates are DNA fragments identical to other ones due to PCR amplification. Unlike genomic duplicates, these reads are not biologically meaningful because they were created artificially through PCR.

[^2]: Optical duplicates are a machine reading error caused by the sequencer, where it incorrectly splits a single read into two or more identical ones.

In [ ]:
dupdata = dat.select(["Sample", "% Duplicates (PCR)", "% Duplicates (Opt)"]).rename(
    {"% Duplicates (PCR)" : "PCR", "% Duplicates (Opt)": "Optical"}
)

def density_chart(col):
    color = "#E69F00" if col == 'PCR' else "#56B4E9"
    return (
        alt.Chart(dupdata, width = 310, height = 250)
        .transform_density(density=col, extent=[0, 100], bandwidth=1)
        .mark_area(opacity=0.9, line=False, fill = color, interpolate="monotone")
        .encode(
            x=alt.X('value:Q', title=None).axis(format='d', labelExpr="datum.value + '%'"),
            y=alt.Y('density:Q', title='Density').axis(labels=False).stack(False),
            tooltip=[
                alt.Tooltip('value:Q', title = "Rate (%)"),
                alt.Tooltip("density:Q", title = "Density", format = '.2f')
            ]
        )
        .properties(title=f"{col} Duplicates")
    ).add_params(alt.selection_interval(bind="scales", encodings=["x"], name=f"brush_{col}"))

with SafeRender(dupdata):
    (
        alt.hconcat(density_chart('PCR'), density_chart('Optical'))
        .properties(
            usermeta={'embedOptions': {'downloadFileName': 'duplication'}}
        )
    ).display()